In [1]:
# Codeblock 1
import torch
import torch.nn as nn

NUM_PRIORS  = 3
NUM_CLASSES = 80

In [4]:
# Codeblock 2
class Convolutional(nn.Module):
    def __init__(self, 
                 in_channels, 
                 out_channels, 
                 kernel_size, 
                 stride=1):
        super().__init__()
        
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels, 
                              kernel_size=kernel_size, 
                              stride=stride,
                              bias=False,    #(1)
                              padding=1 if kernel_size==3 else 0)
        
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        
        self.mish = nn.Mish()                #(2)
        
    def forward(self, x):
        #print(f'original\t: {x.size()}')
        
        x = self.conv(x)
        #print(f'after conv\t: {x.size()}')
        
        x = self.bn(x)
        #print(f'after bn\t: {x.size()}')
        
        x = self.mish(x)
        #print(f'after mish\t: {x.size()}')
        
        return x

In [3]:
# Codeblock 3
convolutional = Convolutional(in_channels=3,
                              out_channels=32,
                              kernel_size=3)

x = torch.randn(1, 3, 416, 416)
out = convolutional(x)

original	: torch.Size([1, 3, 416, 416])
after conv	: torch.Size([1, 32, 416, 416])
after bn	: torch.Size([1, 32, 416, 416])
after mish	: torch.Size([1, 32, 416, 416])


In [7]:
# Codeblock 4
class Residual(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.conv0 = Convolutional(in_channels=num_channels, 
                                   out_channels=num_channels//2,  #(1)
                                   kernel_size=1)                 #(2)
        
        self.conv1 = Convolutional(in_channels=num_channels//2,
                                   out_channels=num_channels,     #(3)
                                   kernel_size=3)                 #(4)
        
    def forward(self, x):
        original = x.clone()
        #print(f'original\t: {x.size()}')
        
        x = self.conv0(x)
        #print(f'after conv0\t: {x.size()}')
        
        x = self.conv1(x)
        #print(f'after conv1\t: {x.size()}')
        
        x = x + original
        #print(f'after summation\t: {x.size()}')
        
        return x

In [6]:
# Codeblock 5
residual = Residual(num_channels=64)

x = torch.randn(1, 64, 208, 208)
out = residual(x)

original	: torch.Size([1, 64, 208, 208])
after conv0	: torch.Size([1, 32, 208, 208])
after conv1	: torch.Size([1, 64, 208, 208])
after summation	: torch.Size([1, 64, 208, 208])


In [10]:
# Codeblock 6
class CSPResidual(nn.Module):
    def __init__(self, 
                 in_channels, 
                 num_blocks):
        super().__init__()
        
        mid_channels = in_channels
        out_channels = in_channels*2
        
        self.downsample = Convolutional(in_channels, 
                                        out_channels, 
                                        kernel_size=3, 
                                        stride=2)      #(1)
        
        self.split0 = Convolutional(out_channels, 
                                    mid_channels, 
                                    kernel_size=1)
        
        self.split1 = Convolutional(out_channels, 
                                    mid_channels, 
                                    kernel_size=1)
        
        self.residuals = nn.ModuleList([Residual(num_channels=mid_channels) for _ in range(num_blocks)])
        
        self.transition0 = Convolutional(mid_channels, 
                                         mid_channels, 
                                         kernel_size=1)
        
        self.transition1 = Convolutional(out_channels, 
                                         out_channels, 
                                         kernel_size=1)
        
    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        x = self.downsample(x)    #(2)
        #print(f'x after downsample\t: {x.size()}')
        
        x0 = self.split0(x)       #(3)
        #print(f'after split0 (x0)\t: {x0.size()}')
        
        x1 = self.split1(x)       #(4)
        #print(f'after split1 (x1)\t: {x1.size()}')
        
        for i, residual in enumerate(self.residuals):    #(5)
            x1 = residual(x1)
            #print(f'x1 after residual #{i}\t: {x1.size()}')
        
        x1 = self.transition0(x1) #(6)
        #print(f'x1 after transition0\t: {x1.size()}')
        
        x = torch.cat([x1, x0], dim=1)
        #print(f'x after concatenate\t: {x.size()}')
        
        x = self.transition1(x)   #(7)
        #print(f'x after transition1\t: {x.size()}')
        
        return x

In [9]:
# Codeblock 7
cspresidual = CSPResidual(in_channels=512, num_blocks=4)

x = torch.randn(1, 512, 13, 13)
out = cspresidual(x)

original		: torch.Size([1, 512, 13, 13])
x after downsample	: torch.Size([1, 1024, 7, 7])
after split0 (x0)	: torch.Size([1, 512, 7, 7])
after split1 (x1)	: torch.Size([1, 512, 7, 7])
x1 after residual #0	: torch.Size([1, 512, 7, 7])
x1 after residual #1	: torch.Size([1, 512, 7, 7])
x1 after residual #2	: torch.Size([1, 512, 7, 7])
x1 after residual #3	: torch.Size([1, 512, 7, 7])
x1 after transition0	: torch.Size([1, 512, 7, 7])
x after concatenate	: torch.Size([1, 1024, 7, 7])
x after transition1	: torch.Size([1, 1024, 7, 7])


In [13]:
# Codeblock 8
class CSPDarknet53(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.first_conv = Convolutional(in_channels=3,             #(1)
                                        out_channels=32, 
                                        kernel_size=3)
        self.stage0 = CSPResidual(in_channels=32, num_blocks=1)    #(2)
        self.stage1 = CSPResidual(in_channels=64, num_blocks=2)
        self.stage2 = CSPResidual(in_channels=128, num_blocks=8)
        self.stage3 = CSPResidual(in_channels=256, num_blocks=8)
        self.stage4 = CSPResidual(in_channels=512, num_blocks=4)   #(3)
        
    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        x   = self.first_conv(x)
        #print(f'after first conv (x)\t: {x.size()}')
        
        x2  = self.stage0(x)
        #print(f'after stage0 (x2)\t: {x2.size()}')
        
        x4  = self.stage1(x2)
        #print(f'after stage1 (x4)\t: {x4.size()}')
        
        x8  = self.stage2(x4)
        #print(f'after stage2 (x8)\t: {x8.size()}')
        
        x16 = self.stage3(x8)
        #print(f'after stage3 (x16)\t: {x16.size()}')
        
        x32 = self.stage4(x16)
        #print(f'after stage4 (x32)\t: {x32.size()}')
        
        return x8, x16, x32      #(4)

In [12]:
# Codeblock 9
cspdarknet53 = CSPDarknet53()
x = torch.randn(1, 3, 416, 416)

out = cspdarknet53(x)

original		: torch.Size([1, 3, 416, 416])
after first conv (x)	: torch.Size([1, 32, 416, 416])
after stage0 (x2)	: torch.Size([1, 64, 208, 208])
after stage1 (x4)	: torch.Size([1, 128, 104, 104])
after stage2 (x8)	: torch.Size([1, 256, 52, 52])
after stage3 (x16)	: torch.Size([1, 512, 26, 26])
after stage4 (x32)	: torch.Size([1, 1024, 13, 13])


In [16]:
# Codeblock 10
class SPP(nn.Module):
    def __init__(self, 
                 in_channels=1024, 
                 out_channels=512):
        super().__init__()
        
        self.preconv0 = Convolutional(in_channels, out_channels, kernel_size=1)
        self.preconv1 = Convolutional(out_channels, in_channels, kernel_size=3)
        self.preconv2 = Convolutional(in_channels, out_channels, kernel_size=1)
        
        self.maxpools = nn.ModuleList([
            nn.MaxPool2d(kernel_size=5,  stride=1, padding=2),    #(1)
            nn.MaxPool2d(kernel_size=9,  stride=1, padding=4),
            nn.MaxPool2d(kernel_size=13, stride=1, padding=6)     #(2)
        ])
        
        self.postconv0 = Convolutional(out_channels*4, out_channels, kernel_size=1)
        self.postconv1 = Convolutional(out_channels, in_channels, kernel_size=3)
        self.postconv2 = Convolutional(in_channels, out_channels, kernel_size=1)
    
    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        x = self.preconv0(x)
        #print(f'after preconv0\t\t: {x.size()}')
        
        x = self.preconv1(x)
        #print(f'after preconv1\t\t: {x.size()}')
        
        x = self.preconv2(x)
        #print(f'after preconv2\t\t: {x.size()}\n')
        
        pooled_outputs = []                                  #(3)
        pooled_outputs.append(x)                             #(4)
        
        for maxpool in self.maxpools:
            pooled = maxpool(x)
            #print(f'after pool\t\t: {pooled.size()}')
            
            pooled_outputs.append(pooled)                    #(5)
            
        concatenated = torch.cat(pooled_outputs, dim=1)      #(6)
        #print(f'after concat\t\t: {concatenated.size()}\n')

        
        x = self.postconv0(concatenated)
        #print(f'after postconv0\t\t: {x.size()}')
        
        x = self.postconv1(x)
        #print(f'after postconv1\t\t: {x.size()}')
        
        x = self.postconv2(x)
        #print(f'after postconv2\t\t: {x.size()}')
        
        return x

In [15]:
# Codeblock 11
spp = SPP()
x32 = torch.randn(1, 1024, 13, 13)

out = spp(x32)

original		: torch.Size([1, 1024, 13, 13])
after preconv0		: torch.Size([1, 512, 13, 13])
after preconv1		: torch.Size([1, 1024, 13, 13])
after preconv2		: torch.Size([1, 512, 13, 13])

after pool		: torch.Size([1, 512, 13, 13])
after pool		: torch.Size([1, 512, 13, 13])
after pool		: torch.Size([1, 512, 13, 13])
after concat		: torch.Size([1, 2048, 13, 13])

after postconv0		: torch.Size([1, 512, 13, 13])
after postconv1		: torch.Size([1, 1024, 13, 13])
after postconv2		: torch.Size([1, 512, 13, 13])


In [19]:
# Codeblock 12
class FiveConvs(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        
        out_channels = in_channels // 2
        
        self.convs = nn.ModuleList([
            Convolutional(in_channels, out_channels, kernel_size=1),
            Convolutional(out_channels, in_channels, kernel_size=3),
            Convolutional(in_channels, out_channels, kernel_size=1),
            Convolutional(out_channels, in_channels, kernel_size=3),
            Convolutional(in_channels, out_channels, kernel_size=1)
        ])
        
    def forward(self, x):
        #print(f'original\t: {x.size()}')
        
        for i, conv in enumerate(self.convs):
            x = conv(x)
            #print(f'after conv #{i}\t: {x.size()}')
        
        return x

In [18]:
# Codeblock 13
fiveconvs = FiveConvs(in_channels=512)
x = torch.randn(1, 512, 26, 26)

out = fiveconvs(x)

original	: torch.Size([1, 512, 26, 26])
after conv #0	: torch.Size([1, 256, 26, 26])
after conv #1	: torch.Size([1, 512, 26, 26])
after conv #2	: torch.Size([1, 256, 26, 26])
after conv #3	: torch.Size([1, 512, 26, 26])
after conv #4	: torch.Size([1, 256, 26, 26])


In [20]:
# Codeblock 14
class PAN(nn.Module):
    def __init__(self):
        super().__init__()
        
        ## Layers for top-down flow.
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        
        self.conv_x32_to_x16        = Convolutional(in_channels=512, out_channels=256, kernel_size=1)
        self.conv_lateral_x16       = Convolutional(in_channels=512, out_channels=256, kernel_size=1)
        self.conv_combined_x16_td   = FiveConvs(in_channels=512)
        
        self.conv_x16_to_x8         = Convolutional(in_channels=256, out_channels=128, kernel_size=1)
        self.conv_lateral_x8        = Convolutional(in_channels=256, out_channels=128, kernel_size=1)
        self.conv_combined_x8_td    = FiveConvs(in_channels=256)
        
        ## Layers for bottom-up flow.
        self.conv_x8_to_x16       = Convolutional(in_channels=128, out_channels=256, kernel_size=3, stride=2)
        self.conv_combined_x16_bu = FiveConvs(in_channels=512)
        
        self.conv_x16_to_x32      = Convolutional(in_channels=256, out_channels=512, kernel_size=3, stride=2)
        self.conv_combined_x32_bu = FiveConvs(in_channels=1024)
        

    def forward(self, x8, x16, x32):
        
        ## Top-down flow.
        x32_upsampled = self.upsample(self.conv_x32_to_x16(x32))
        x16_lateral = self.conv_lateral_x16(x16)
        x16_concatenated_td = torch.cat([x32_upsampled, x16_lateral], dim=1)
        x16_combined_td = self.conv_combined_x16_td(x16_concatenated_td)
        
        x16_upsampled = self.upsample(self.conv_x16_to_x8(x16_combined_td))
        x8_lateral = self.conv_lateral_x8(x8)
        x8_concatenated_td = torch.cat([x16_upsampled, x8_lateral], dim=1)
        x8_combined_td = self.conv_combined_x8_td(x8_concatenated_td)
        
        x8_out = x8_combined_td
        
        ## Bottom-up flow.
        x8_downsampled = self.conv_x8_to_x16(x8_combined_td)
        x16_concatenated_bu = torch.cat([x8_downsampled, x16_combined_td], dim=1)
        x16_combined_bu = self.conv_combined_x16_bu(x16_concatenated_bu)
        
        x16_out = x16_combined_bu
        
        x16_downsampled = self.conv_x16_to_x32(x16_combined_bu)
        x32_concatenated_bu = torch.cat([x16_downsampled, x32], dim=1)
        x32_combined_bu = self.conv_combined_x32_bu(x32_concatenated_bu)
        
        x32_out = x32_combined_bu
        
        return x8_out, x16_out, x32_out

In [21]:
# Codeblock 15
pan = PAN()
x8  = torch.randn(1, 256, 52, 52)
x16 = torch.randn(1, 512, 26, 26)    #(1)
x32 = torch.randn(1, 512, 13, 13)    #(2)

out = pan(x8, x16, x32)

print(out[0].shape)
print(out[1].shape)
print(out[2].shape)

torch.Size([1, 128, 52, 52])
torch.Size([1, 256, 26, 26])
torch.Size([1, 512, 13, 13])


In [24]:
# Codeblock 16
class SAM(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=num_channels,    #(1)
                              out_channels=num_channels,
                              kernel_size=1, 
                              stride=1,
                              bias=False)
        self.batchnorm = nn.BatchNorm2d(num_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        attention = self.conv(x)
        #print(f'attention after conv\t: {attention.size()}')
        
        attention = self.batchnorm(attention)    #(2)
        #print(f'attention after bn\t: {attention.size()}')
        
        attention = self.sigmoid(attention)      #(3)
        #print(f'attention after sigmoid\t: {attention.size()}')
        
        x = x * attention                        #(4)
        #print(f'weighted tensor\t\t: {x.size()}')
        
        return x

In [23]:
# Codeblock 17
sam = SAM(num_channels=512)
x32 = torch.randn(1, 512, 13, 13)

out = sam(x32)

original		: torch.Size([1, 512, 13, 13])
attention after conv	: torch.Size([1, 512, 13, 13])
attention after bn	: torch.Size([1, 512, 13, 13])
attention after sigmoid	: torch.Size([1, 512, 13, 13])
weighted tensor		: torch.Size([1, 512, 13, 13])


In [27]:
# Codeblock 18
class YOLOv3Head(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.conv = Convolutional(in_channels=num_channels, 
                                  out_channels=num_channels*2,
                                  kernel_size=3)
        
        self.pred = nn.Conv2d(in_channels=num_channels*2, 
                              out_channels=NUM_PRIORS*(5+NUM_CLASSES),
                              kernel_size=1)

    def forward(self, x):
        #print(f'original\t : {x.size()}')
        
        x = self.conv(x)
        #print(f'after conv\t : {x.size()}')
        
        x = self.pred(x)
        #print(f'final prediction : {x.size()}')
        
        return x

In [26]:
# Codeblock 19
yolov3head = YOLOv3Head(num_channels=512)
x32 = torch.randn(1, 512, 13, 13)

out = yolov3head(x32)

original	 : torch.Size([1, 512, 13, 13])
after conv	 : torch.Size([1, 1024, 13, 13])
final prediction : torch.Size([1, 255, 13, 13])


In [28]:
# Codeblock 20
class YOLOv4(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.cspdarknet53 = CSPDarknet53()            #(1)
        self.spp = SPP()                              #(2)
        self.pan = PAN()                              #(3)
        
        self.sam_s = SAM(num_channels=128)            #(4)
        self.sam_m = SAM(num_channels=256)
        self.sam_l = SAM(num_channels=512)
        
        self.head_s = YOLOv3Head(num_channels=128)
        self.head_m = YOLOv3Head(num_channels=256)
        self.head_l = YOLOv3Head(num_channels=512)    #(5)
        
    def forward(self, x):                             #(6)
        print(f'x original\t\t: {x.size()}\n')
        
        x8, x16, x32 = self.cspdarknet53(x)           #(7)
        
        print(f'x8 after backbone\t: {x8.size()}')
        print(f'x16 after backbone\t: {x16.size()}')
        print(f'x32 after backbone\t: {x32.size()}\n')
        
        x32 = self.spp(x32)                           #(8)
        print(f'x32 after spp\t\t: {x32.size()}\n')
        
        x8, x16, x32 = self.pan(x8, x16, x32)         #(9)
        print(f'x8 after panet\t\t: {x8.size()}')
        print(f'x16 after panet\t\t: {x16.size()}')
        print(f'x32 after panet\t\t: {x32.size()}\n')
        
        x8  = self.sam_s(x8)                          #(10)
        x16 = self.sam_m(x16)
        x32 = self.sam_l(x32)                         #(11)
        print(f'x8 after sam_s\t\t: {x8.size()}')
        print(f'x16 after sam_m\t\t: {x16.size()}')
        print(f'x32 after sam_l\t\t: {x32.size()}\n')
        
        x8  = self.head_s(x8)                         #(12)
        x16 = self.head_m(x16)
        x32 = self.head_l(x32)                        #(13)
        print(f'x8 after head_s\t\t: {x8.size()}')
        print(f'x16 after head_m\t: {x16.size()}')
        print(f'x32 after head_l\t: {x32.size()}')
        
        return x8, x16, x32                           #(14)

In [29]:
# Codeblock 21
yolov4 = YOLOv4()
x = torch.randn(1, 3, 416, 416)

out = yolov4(x)

x original		: torch.Size([1, 3, 416, 416])

x8 after backbone	: torch.Size([1, 256, 52, 52])
x16 after backbone	: torch.Size([1, 512, 26, 26])
x32 after backbone	: torch.Size([1, 1024, 13, 13])

x32 after spp		: torch.Size([1, 512, 13, 13])

x8 after panet		: torch.Size([1, 128, 52, 52])
x16 after panet		: torch.Size([1, 256, 26, 26])
x32 after panet		: torch.Size([1, 512, 13, 13])

x8 after sam_s		: torch.Size([1, 128, 52, 52])
x16 after sam_m		: torch.Size([1, 256, 26, 26])
x32 after sam_l		: torch.Size([1, 512, 13, 13])

x8 after head_s		: torch.Size([1, 255, 52, 52])
x16 after head_m	: torch.Size([1, 255, 26, 26])
x32 after head_l	: torch.Size([1, 255, 13, 13])
